In [1]:
# Setup your LLM keys
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [3]:
from crewai import Agent, Task, Crew, LLM, Process
from crewai_tools import SerperDevTool

In [4]:
llm = LLM(model="gpt-4o-mini")

In [5]:
incident_description = """
Users were unable to log in between 10:00 AM and 11:30 AM.
Authentication requests failed and API response times increased significantly.
Multiple users reported timeout errors.
"""

## Incident Analyst Agent

In [6]:
incident_analyst = Agent(
    role="Incident Analyst",
    goal="Analyze production incidents and summarize findings",
    backstory="""An experienced Site Reliability Engineer (SRE) who investigates system outages and identifies key symptoms and impacts.""",
    llm=llm,
    verbose=True
)

## Root Cause Engineer Agent

In [7]:
root_cause_engineer = Agent(
    role="Root Cause Engineer",
    goal="Identify probable root causes and recommend corrective actions",
    backstory="""A senior platform engineer specializing in troubleshooting, reliability engineering, and incident prevention.""",
    llm=llm,
    verbose=True
)

## Task-1 Incident Analysis Task

In [8]:
incident_analysis_task = Task(
    description=f"""Analyze the following incident report. Incident Details:
    {incident_description}

    Provide:
    1. Incident Summary
    2. Business Impact
    3. Affected Components
    4. Key Observations
    """,
    expected_output="""A structured incident analysis report including summary,impact assessment, affected services, and observations.""",
    agent=incident_analyst
)

## Task 2 - Root Cause Task

In [9]:
root_cause_task = Task(
    description=""" Based on the incident analysis report:
    Provide:
    1. Probable Root Cause(s)
    2. Supporting Evidence
    3. Immediate Remediation Actions
    4. Long-Term Preventive Measures
    """,
    expected_output=""" A root cause analysis report with remediation and prevention recommendations.""",
    agent=root_cause_engineer,
    context=[incident_analysis_task]
)

## Crew

In [11]:
crew = Crew(agents=[incident_analyst,root_cause_engineer],
            tasks=[incident_analysis_task,root_cause_task],
            process=Process.sequential,
            verbose=True
)

In [12]:
result = crew.kickoff()

print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 105c2c58-6808-4815-9a75-cbe2aea001d5                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Incident Analyst                                                                                        │
│                                                                                                                 │
│  Task: Analyze the following incident report. Incident Details:                                                 │
│                                                                                                                 │
│  Users were unable to log in between 10:00 AM and 11:30 AM.                                                     │
│  Authentication requests failed and API response times increased significantly.                                 │
│  Multiple users reported timeout errors.                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Provide:                                                                                                   │
│      1. Incident Summary                                                                                        │
│      2. Business Impact                                                                                         │
│      3. Affected Components                                                                                     │
│      4. Key Observations                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

14:16:06 - LiteLLM:INFO: utils.py:3258 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai


Output()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
14:16:13 - LiteLLM:INFO: utils.py:1260 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Incident Analyst                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Incident Summary**:                                                                                          │
│  On [insert date], users encountered login issues from 10:00 AM to 11:30 AM. During this window,                │
│  authentication requests were failing, leading to significant delays in API response times. Multiple users      │
│  reported experiencing timeout errors, preventing them from accessing their accounts and utilizing the          │
│  system's functionalities.                                                                                      │
│                                                                                                                 │
│  **Business Impact**:                                                                                           │
│  The inability for users to log in directly impacted business operations, resulting in customer                 │
│  dissatisfaction and potential revenue loss. Key transactions were unable to be completed during the outage,    │
│  and service-level agreements (SLAs) may have been breached, potentially damaging customer trust and brand      │
│  reputation. Furthermore, support teams were likely overwhelmed with user reports, diverting resources from     │
│  other operational priorities.                                                                                  │
│                                                                                                                 │
│  **Affected Components**:                                                                                       │
│  1. Authentication Service: The primary service responsible for user logins was unable to process requests      │
│  successfully, leading to widespread login failures.                                                            │
│  2. API Gateway: The interface responsible for directing authentication requests showed increased response      │
│  times, indicating performance degradation.                                                                     │
│  3. User Database: Potentially underlying issues with database connectivity or performance could have           │
│  contributed to the failures in authentication requests.                                                        │
│                                                                                                                 │
│  **Key Observations**:                                                                                          │
│  - The peak of issues occurred between 10:00 AM and 11:30 AM, indicating a possible correlation to changes or   │
│  deployments made earlier in the day.                                                                           │
│  - Increased API response times were noted, suggesting that backend services experienced strain under the load  │
│  or that there were delays in processing requests.                                                              │
│  - The pattern of timeout errors across multiple users indicates a systemic issue rather than isolated          │
│  incidents, pointing to a common failure point in the authentication workflow.                                  │
│  - Analysis of server logs during the incident window revealed spikes in error rates related to authentication  │
│  attempts and database queries, highlighting possible b

... (5 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Root Cause Engineer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Root Cause Analysis Report**                                                                                 │
│                                                                                                                 │
│  **1. Probable Root Cause(s):**                                                                                 │
│  - **Authentication Service Deterioration**: The primary cause of the login issues appears to be a degradation  │
│  in the authentication service. This could arise from software bugs introduced by recent changes or a           │
│  misconfiguration in the service settings.                                                                      │
│  - **Database Connectivity Issues**: The user database’s performance likely suffered due to connection pool     │
│  exhaustion or inefficient query execution, contributing to the timeout errors users faced while trying to log  │
│  in.                                                                                                            │
│  - **Increased Load on API Gateway**: The API Gateway, responsible for directing authentication requests, may   │
│  have been under increased load beyond its handling capacity during peak hours, which could have exacerbated    │
│  response times.                                                                                                │
│                                                                                                                 │
│  **2. Supporting Evidence:**                                                                                    │
│  - **Time Correlation with Changes**: The spike in incidents coincided with deployments conducted earlier in    │
│  the day, suggesting that the introduction of new features or changes may have destabilized the authentication  │
│  process.                                                                                                       │
│  - **Server Logs Review**: Analysis of server logs showed elevated error rates in authentication attempts and   │
│  significant delays in database queries during the incident window, pointing directly to the affected           │
│  components.                                                                                                    │
│  - **User Reports**: The collective feedback from multiple users experiencing similar timeout errors            │
│  reaffirmed the existence of a systemic issue, rather than isolated failures.                                   │
│                                                                                                                 │
│  **3. Immediate Remediation Actions:**                                                                          │
│  - **Rollback Recent Changes**: Immediately rollback any recent changes or deployments made to the              │
│  authentication service and API Gateway until further analysis can confirm stability.                           │
│  - **Database Connection Review**: Check and optimize database connection settings to ensure the connection     │
│  pool has sufficient resources to handle the peak load, potentially increasing the pool size temporarily to     │
│  alleviate strain.                                                                                              │
│  - **Enable API Rate Limiting**: Temporarily implement 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: f328bfa9-2d63-45cc-8201-49507019e1f0                                                                     │
│  Agent: Root Cause Engineer                                                                                     │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 105c2c58-6808-4815-9a75-cbe2aea001d5                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Root Cause Analysis Report**                                                                   │
│                                                                                                                 │
│  **1. Probable Root Cause(s):**                                                                                 │
│  - **Authentication Service Deterioration**: The primary cause of the login issues appears to be a degradation  │
│  in the authentication service. This could arise from software bugs introduced by recent changes or a           │
│  misconfiguration in the service settings.                                                                      │
│  - **Database Connectivity Issues**: The user database’s performance likely suffered due to connection pool     │
│  exhaustion or inefficient query execution, contributing to the timeout errors users faced while trying to log  │
│  in.                                                                                                            │
│  - **Increased Load on API Gateway**: The API Gateway, responsible for directing authentication requests, may   │
│  have been under increased load beyond its handling capacity during peak hours, which could have exacerbated    │
│  response times.                                                                                                │
│                                                                                                                 │
│  **2. Supporting Evidence:**                                                                                    │
│  - **Time Correlation with Changes**: The spike in incidents coincided with deployments conducted earlier in    │
│  the day, suggesting that the introduction of new features or changes may have destabilized the authentication  │
│  process.                                                                                                       │
│  - **Server Logs Review**: Analysis of server logs showed elevated error rates in authentication attempts and   │
│  significant delays in database queries during the incident window, pointing directly to the affected           │
│  components.                                                                                                    │
│  - **User Reports**: The collective feedback from multiple users experiencing similar timeout errors            │
│  reaffirmed the existence of a systemic issue, rather than isolated failures.                                   │
│                                                                                                                 │
│  **3. Immediate Remediation Actions:**                                                                          │
│  - **Rollback Recent Changes**: Immediately rollback any recent changes or deployments made to the              │
│  authentication service and API Gateway until further analysis can confirm stability.                           │
│  - **Database Connection Review**: Check and optimize database connection settings to ensure the connection     │
│  pool has sufficient resources to handle the peak load, potentially increasing the pool size temporarily to     │
│  alleviate strain.                                    

**Root Cause Analysis Report**

**1. Probable Root Cause(s):**  
- **Authentication Service Deterioration**: The primary cause of the login issues appears to be a degradation in the authentication service. This could arise from software bugs introduced by recent changes or a misconfiguration in the service settings.  
- **Database Connectivity Issues**: The user database’s performance likely suffered due to connection pool exhaustion or inefficient query execution, contributing to the timeout errors users faced while trying to log in.  
- **Increased Load on API Gateway**: The API Gateway, responsible for directing authentication requests, may have been under increased load beyond its handling capacity during peak hours, which could have exacerbated response times.

**2. Supporting Evidence:**  
- **Time Correlation with Changes**: The spike in incidents coincided with deployments conducted earlier in the day, suggesting that the introduction of new features or changes may have destabi